In [1]:
version = "REPLACE_PACKAGE_VERSION"

## Experiment Design and Analysis
## School of Information, University of Michigan

## Week 1: 
- 1. What are experiments?
- 2. Experimental Design
- 3. Lab vs. Field Experiments
- 4. Online Field Experiments

## Assignment Overview
### The objective of this assignment is to:

- Apply theory of experiment design and knowledge of analysis techniques to real experiment data.


### The total score of this assignment will be 10 points

### Resources:
- StatsModels
    - We recommend using a python library called [StatsModels](https://www.statsmodels.org/stable/index.html) for data analysis  
    
    
- Optional Reading: [Holt C.A, & Laury S.K. Risk Aversion and Incentive Effects. (2002).](https://www.jstor.org/stable/3083270)  


- Dataset used in this assignment: Fixed-Price Auction data [download csv file](assignment1_data.csv)
    - Source for dataset: [Chen, Y., et al. Sealed bid auctions with ambiguity: Theory and experiments. (2007).](https://www.sciencedirect.com/science/article/pii/S0022053107000178)

In [2]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.stats.api as sms
from scipy import stats
from mads.lib.path import assets
#you may or may not use all of the above libraries, and that is OK!
file = assets.find("assignment1_data.csv")
data = pd.read_csv(file) #Data for this assignment

In [3]:
#uncomment the below line to view readme file for this dataset (includes explanation of variable names)
file_readme = assets.find("assignment1_data_readme.md")
!cat $file_readme

#uncomment the below line to view snippet of csv file
data.head()

### Assignment Topic: Data analysis of a laboratory experiment on first-price auction

### Background:
We upload data files from a laboratory experiment conducted at the University of Michigan.

There are ten experimental sessions, with eight subjects per session. In this context, subjects are tasked with completing auction and lottery (Holt-Laury 2002) tasks in two orders.

   - In five of the ten sessions, subjects first complete a lottery task, followed by 30 rounds of auctions.
   - In the other five sessions, subjects first complete 30 rounds of auctions, followed by a lottery task.

At the end of each session, subjects complete a demographics survey.

### Data:

The data has the following variables:
   - treatment: the treatment received by the subject
   - session: the session in which the data was collected in the experiment
   - period: the period in the session (multiple periods per session)
   - subject: unique identifier for each subject
   - disttype: distribution type, hi

,treatment,session,period,subject,disttype,highdist,lowdist,group,v,b,...,irritation,moodswings,withdrawal,major,sdmajor,major1,major2,major3,major4,major5
0,k1_8_exp_lot,061018_1,1,1,Low,0,1,1,48,40,...,0,0,0,2,Electrical Engineering - Signal Processing (st...,0,1,0,0,0
1,k1_8_exp_lot,061018_1,1,2,High,1,0,4,76,15,...,0,0,0,4,public health,0,0,0,1,0
2,k1_8_exp_lot,061018_1,1,3,High,1,0,3,73,53,...,0,0,0,5,german and film and video studies,0,0,0,0,1
3,k1_8_exp_lot,061018_1,1,4,High,1,0,4,74,51,...,1,0,0,5,spanish,0,0,0,0,1
4,k1_8_exp_lot,061018_1,1,5,Low,0,1,1,72,52,...,0,1,0,2,Engineering,0,1,0,0,0


## Part A (6 points)

Suppose subjects were randomly assigned to two treatment groups. We want to know if the randomization was properly applied to these groups. In other words, we want to know if the proportion of participants in these demographic groups are different between the two treatments.

1. To determine if the randomization worked, for each of the two treatments, modify the following ```stats_calculator``` function so that it can input a dataframe and tabulate the mean, standard deviation, minimum and maximum of the following variables: female, age, number of siblings, white, asian, african american, hispanic, and other ethnicities. (6 points)


In [15]:
def stats_calculator(provided_data):

    # create empty dataframe with required output columns
    stats_df = pd.DataFrame(columns=['variable','mean','std. dev.','max','min'])
    
    # list of demo variables required
    variables = ['female','age','siblings','white','asian','african','hispanic','other']
    stats_df['variable'] = variables
    
    # loop through each variable and calc summary stats
    for variable in stats_df['variable']:
        # calc the mean and ound to 2 decimal places
        stats_df.loc[stats_df['variable']==variable,'mean'] = round(provided_data[variable].mean(), 2)
        
        # calc stand deviation and round to 2 decimal places
        stats_df.loc[stats_df['variable']==variable, 'std. dev.'] = round(provided_data[variable].std(), 2)
        
        # find the max value and round to 2 decimal places
        stats_df.loc[stats_df['variable']==variable, 'max'] = round(provided_data[variable].max(), 2)
        
        # find the min value and round to 2 decimal places
        stats_df.loc[stats_df['variable']==variable, 'min'] = round(provided_data[variable].min(), 2)
        
    # set variable as the dataframe index to match the required output frame
    stats_df.set_index('variable', inplace= True)
    
    # return completed summary stats dataframe
    return stats_df

We can use the function to check that the statistics for the control group are similar to those of the treatment group. In order to do that, we will need one dataframe for the control and one for the treatment group. Those are created in the next cell.

In [16]:
control = data[data['treatment'] == 'k1_8_lot_exp']
treatment = data[data['treatment'] == 'k1_8_exp_lot']

In [17]:
stats_calculator(control)

,mean,std. dev.,max,min
variable,,,,
female,0.62,0.49,1,0
age,21.8,3.31,31,18
siblings,1.68,1.12,4,0
white,0.46,0.5,1.0,0.0
asian,0.29,0.45,1.0,0.0
african,0.12,0.33,1.0,0.0
hispanic,0.05,0.22,1.0,0.0
other,0.08,0.27,1,0


In [18]:
stats_calculator(treatment)

,mean,std. dev.,max,min
variable,,,,
female,0.65,0.48,1,0
age,23.22,3.55,31,19
siblings,1.6,1.3,5,0
white,0.48,0.51,1.0,0.0
asian,0.25,0.42,1.0,0.0
african,0.1,0.28,1.0,0.0
hispanic,0.1,0.28,1.0,0.0
other,0.08,0.27,1,0


In [19]:
"""Check that the function outputs the correct (rounded) statistics"""
assert stats_calculator(control).loc['female']['mean'] == 0.62, "Part A #1 female mean value differs"
assert stats_calculator(treatment).loc['age']['std. dev.'] == 3.55, "Part A #1 age std. dev value differs"

In [20]:
"""Hidden Tests: Check function outputs correct (rounded) statistics for control group"""
# Hidden tests

'Hidden Tests: Check function outputs correct (rounded) statistics for control group'

In [21]:
"""Hidden Tests: Check function outputs correct (rounded) statistics for treatment group"""
# Hidden tests

'Hidden Tests: Check function outputs correct (rounded) statistics for treatment group'

## Part B (4 points)

We can also use a more objective measure to identify if our treatment groups were properly randomized.

1. Using a __t-test__ (make sure you use the _correct_ type of t-test) and the ```data``` dataframe again, analyze the differences between the two treatment groups (__k1_8_exp_lot__ and __k1_8_lot_exp__) for the female, age, and hispanic demographic variables by completing the following ```objective_randomization``` function. (4 points)

**Round any calculations to the hundredth decimal. Do not use percentages.**

In [22]:
def objective_randomization(provided_data):
    # create empty dataframe with required output columns
    ttest_df = pd.DataFrame(columns=['variable','t-statistic','p-value'])

    # variables to compare across treatment groups
    variables = ['female','age','hispanic']
    ttest_df['variable'] = variables

    # loop through each variable and run an independent samples t-test
    for variable in ttest_df['variable']:
        # select observations from the first treatment group
        g1 = provided_data[provided_data['treatment'] == 'k1_8_lot_exp'][variable]

        # select ovbservations from the second treatment group
        g2 = provided_data[provided_data['treatment'] == 'k1_8_exp_lot'][variable]

        # run a two-sample t-test comparing the means of the two groups
        t_stat, p_val = stats.ttest_ind(g1, g2)

        # store the rounded t-statistic in the output dataframe
        ttest_df.loc[ttest_df['variable'] == variable, 't-statistic'] = round(t_stat, 2)

        # store the rounded p-value in the output dataframe
        ttest_df.loc[ttest_df['variable'] == variable, 'p-value'] = round(p_val, 2)
        
    # set variable as the dataframe index to match the required output format
    ttest_df.set_index('variable', inplace= True)

    # return the completed t-test results dataframe
    return ttest_df

Your function should return a dataframe with each of the variables and their completed t-statistic and p-value across the treatments. 

Check that it does:

In [23]:
objective_randomization(data)

,t-statistic,p-value
variable,,
female,-0.23,0.82
age,-1.86,0.07
hispanic,-0.88,0.38


In [24]:
"""Check that the function above outputs the required statistics"""
result = objective_randomization(data)
assert abs(result.loc['female']['t-statistic']) == 0.23, "checking the value of the female t-statistic"

In [25]:
"""Part B # 1: Check function abv outputs required statistics"""
# Hidden tests

'Part B # 1: Check function abv outputs required statistics'